# Clase 110 — Batch Normalization, Layer Normalization

**BatchNorm** (Ioffe & Szegedy 2015) estandariza las activaciones usando estadísticas del batch; **LayerNorm** (Ba et al. 2016) normaliza por muestra y es el default en Transformers y RNN. También **GroupNorm** para batch chico.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`.

## 1. BN antes vs después de la activación

Debate clásico: **antes** suele funcionar mejor con ReLU, **después** con GELU/Swish. Con BN se usa `use_bias=False` en el `Dense` previo.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

def bn_antes():
    return keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(256, use_bias=False, kernel_initializer="he_normal"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dense(10, activation="softmax"),
    ])

def bn_despues():
    return keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(256, activation="relu", kernel_initializer="he_normal"),
        layers.BatchNormalization(),
        layers.Dense(10, activation="softmax"),
    ])

print("BN-antes params:", bn_antes().count_params(),
      "| BN-después params:", bn_despues().count_params())

## 2. Parámetros de BN: `γ`, `β` (trainables) + moving averages

`γ` (scale) y `β` (shift) se aprenden; `moving_mean` y `moving_variance` se acumulan como EMA y se usan en inference.

In [ ]:
bn = layers.BatchNormalization()
bn.build((None, 256))
print("trainable    :", [w.name for w in bn.trainable_weights])       # gamma, beta
print("no trainable :", [w.name for w in bn.non_trainable_weights])   # moving_mean/var

## 3. Train vs inference: las predicciones cambian (correcto)

En `training=True` normaliza con stats del batch; en `training=False` usa las moving averages acumuladas.

In [ ]:
modelo = bn_antes()
x = tf.constant(np.random.default_rng(0).normal(size=(16, 784)), dtype=tf.float32)
p_train = modelo(x, training=True)
p_infer = modelo(x, training=False)
print("¿iguales train vs inference?",
      bool(tf.reduce_all(tf.abs(p_train - p_infer) < 1e-6).numpy()))
print("correcto que difieran: train usa stats del batch, inference las moving averages")

## 4. LayerNorm: normaliza por muestra

Independiente del batch → media ≈ 0 y std ≈ 1 en el eje de features de **cada** muestra.

In [ ]:
ln = layers.LayerNormalization()
x = tf.constant(np.random.default_rng(1).normal(size=(4, 8)), dtype=tf.float32)
y = ln(x)
print("media por muestra tras LN (≈0):", tf.reduce_mean(y, axis=-1).numpy().round(4))
print("std  por muestra tras LN (≈1):", tf.math.reduce_std(y, axis=-1).numpy().round(4))

## 5. LayerNorm en secuencias (RNN / Transformers)

BN falla en RNN (longitudes variables, batch chico rompen las estadísticas). LN es estable.

In [ ]:
secuencia = keras.Sequential([
    keras.Input(shape=(20, 32)),            # (timesteps, features)
    layers.LSTM(32, return_sequences=True),
    layers.LayerNormalization(),            # estable con batch chico y longitudes variables
    layers.LSTM(16),
    layers.Dense(1),
])
print("RNN + LayerNorm (donde BN fallaría):", secuencia.count_params(), "params")

## 6. GroupNorm para batch muy chico

Agrupa canales y normaliza dentro de cada grupo; funciona incluso con `batch_size=1` (segmentación, detección).

In [ ]:
gn = layers.GroupNormalization(groups=8)
gn.build((None, 32))
salida = gn(tf.constant(np.random.default_rng(2).normal(size=(1, 32)), dtype=tf.float32))
print("GroupNorm con 8 grupos, batch_size=1 → shape:", salida.shape)

## Ejercicios

1. **BN vs sin BN**: entrená el mismo MLP con y sin BN; compará épocas hasta `accuracy 0.85`.
2. **Antes vs después**: `Dense→BN→ReLU` vs `Dense→ReLU→BN`; compará.
3. **Inference mode**: entrená con BN y compará `training=True` vs `training=False` sobre un batch.
4. **Batch chico**: forzá `batch_size=4` con BN (inestable) y luego con LayerNorm (estable).

## Conclusiones

- **BN**: `y = γ·(x-μ)/σ + β` con stats del batch en train y moving averages en inference.
- Acelera la convergencia, permite LR más altos y regulariza levemente.
- **LayerNorm** normaliza por muestra: default en RNN/Transformers donde BN falla.
- **GroupNorm** funciona con batch muy chico (visión de segmentación/detección).
- No poner BN justo antes de la softmax; y en custom loops recordá pasar `training=True`.

## ✅ Soluciones de los ejercicios

Batch Normalization vs Layer Normalization: efecto en las curvas, orden respecto a la activación, modo inferencia, batch chico y LayerNorm en secuencias. Sin TF se validan por AST.

**Ej. 1 — BN vs sin BN.** El mismo MLP con y sin BatchNorm: BN converge más rápido y estable.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make(bn):
    L = [keras.Input((784,))]
    for u in (256, 128):
        L.append(layers.Dense(u, use_bias=not bn))
        if bn:
            L.append(layers.BatchNormalization())
        L.append(layers.Activation("relu"))
    L.append(layers.Dense(10, activation="softmax"))
    m = keras.Sequential(L)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for bn in (False, True):
    h = make(bn).fit(Xtr, ytr, epochs=15, validation_split=0.2, verbose=0)
    plt.plot(h.history["val_loss"], label="con BN" if bn else "sin BN")
plt.legend(); plt.xlabel("epoca"); plt.ylabel("val_loss")
plt.title("BatchNorm acelera y estabiliza el entrenamiento"); plt.show()

**Ej. 2 — ¿Antes o después de la activación?** `Dense→BN→ReLU` (receta clásica) vs `Dense→ReLU→BN`.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make(order):
    L = [keras.Input((784,))]
    for u in (256, 128):
        L.append(layers.Dense(u, use_bias=False))
        if order == "bn_first":
            L += [layers.BatchNormalization(), layers.Activation("relu")]
        else:
            L += [layers.Activation("relu"), layers.BatchNormalization()]
    L.append(layers.Dense(10, activation="softmax"))
    m = keras.Sequential(L)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for order in ("bn_first", "act_first"):
    h = make(order).fit(Xtr, ytr, epochs=10, validation_split=0.2, verbose=0)
    print(f"{order}: val_acc = {h.history['val_accuracy'][-1]:.3f}")
print("Ambas funcionan; 'Dense->BN->ReLU' es la del paper (Ioffe & Szegedy, 2015).")

**Ej. 3 — Inference mode.** En `training=False` BN usa las medias/varianzas móviles aprendidas, no las del batch: predicciones deterministas.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, use_bias=False), layers.BatchNormalization(),
    layers.Activation("relu"), layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.fit(Xtr, ytr, epochs=3, verbose=0)
batch = Xtr[:32]
p_train = model(batch, training=True).numpy()    # usa stats del batch
p_infer = model(batch, training=False).numpy()    # usa moving averages
print("¿difieren train vs inference?", not np.allclose(p_train, p_infer))
print("Correcto: en inferencia BN es deterministico (no depende de los otros ejemplos del batch).")

**Ej. 4 — Batch chico → LayerNorm.** Con `batch_size=4`, BN estima mal la estadística; LayerNorm normaliza por muestra y se estabiliza.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make(norm):
    N = layers.BatchNormalization if norm == "bn" else layers.LayerNormalization
    m = keras.Sequential([keras.Input((784,)),
        layers.Dense(128, use_bias=False), N(), layers.Activation("relu"),
        layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for norm in ("bn", "ln"):
    h = make(norm).fit(Xtr, ytr, epochs=5, batch_size=4, validation_split=0.2, verbose=0)
    print(f"{norm}: val_acc = {h.history['val_accuracy'][-1]:.3f}")
print("BN con batch=4 es inestable; LayerNorm no depende del tamano de batch.")

**Ej. 5 — LayerNorm en RNN.** `LayerNormalization` previo a una `LSTM` — el estándar en secuencias y Transformers.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

inp = keras.Input(shape=(20, 16))                 # (timesteps, features)
x = layers.LayerNormalization()(inp)
x = layers.LSTM(32, recurrent_activation="sigmoid")(x)
out = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inp, out)
model.compile(optimizer="adam", loss="binary_crossentropy")
model.summary()
print("LayerNorm normaliza por timestep/feature: no depende del batch, ideal para secuencias.")